# Multimodal — โจทย์แบบ "ตอบคำถามจากรูป / อ่านข้อความในรูป" (VQA / OCR)

รูป (+ คำถาม) -> คำตอบสั้น ๆ เช่น "ในรูปมีกี่คน", "ป้ายเขียนว่าอะไร"

วิธีในโน้ตบุ๊กนี้:
- วิธีที่ 1 (ง่ายสุด) = VLM สำเร็จรูปแบบ zero-shot (BLIP-VQA / Qwen2.5-VL) ไม่ต้องเทรน
- วิธีที่ 2 (ขั้นสูง) = fine-tune VLM (ดู reference_cheatsheet.md)


## เมื่อไหร่ใช้โน้ตบุ๊กนี้

ใช้เมื่อมี "รูป + คำถาม" -> ตอบ หรือ "อ่านตัวอักษรในรูป (OCR)"
ถ้าแค่บรรยายรูป (ไม่มีคำถาม) -> `thai_image_captioning.ipynb`

ต้องแก้ (`# << แก้`): ชื่อ competition, path โฟลเดอร์รูป, คอลัมน์คำถาม `Q_COL`, คอลัมน์ id รูป

## ขั้นที่ 1 — ติดตั้ง

In [ ]:
!pip -q install -U transformers pillow pandas torch
!pip -q install pythainlp   # เผื่อวัด/ตัดคำไทย

## ขั้นที่ 2 — เอาข้อมูลเข้า (เลือก 1 ใน 3 วิธี) + เช็ค GPU

เซลล์ล่างรองรับ 3 วิธี แก้แค่ตัวแปรบนสุดให้ตรงกับวิธีที่ใช้:

วิธี A (แนะนำ) ดาวน์โหลดจาก Kaggle อัตโนมัติ: ต้องมี `kaggle.json` (kaggle.com -> รูปโปรไฟล์ -> Settings -> Account -> Create New Token)
- บน `Kaggle Notebook`: ข้อมูลต่อไว้ที่ `/kaggle/input/...` แล้ว ไม่ต้องใส่ token
- บน `Colab/เครื่อง`: ใส่ `KAGGLE_USERNAME` + `KAGGLE_KEY`

วิธี B โหลดข้อมูลมาเครื่องตัวเองแล้วอัปขึ้น Colab เอง: ลากไฟล์ (เช่น `data.zip`) ไปวางในแถบ Files ซ้ายมือของ Colab
แล้วตั้ง `DATA_DIR = "/content"` (หรือโฟลเดอร์ที่วาง) -> เซลล์จะแตก zip ให้เอง ไม่ต้องใช้ token

วิธี C ต่อ Google Drive: รัน `from google.colab import drive; drive.mount('/content/drive')` ก่อน แล้วตั้ง `DATA_DIR = "/content/drive/MyDrive/โฟลเดอร์ของคุณ"`

เซลล์นี้เช็ค GPU ให้ด้วย ถ้างานเป็น deep learning (รูป/ข้อความ/สัญญาณดิบ) ควรเปิด GPU: เมนู `Runtime > Change runtime type > T4 GPU`

In [ ]:
import os, glob, subprocess

# ----- วิธี A: ดาวน์โหลดจาก Kaggle -----
COMP = "ใส่-slug-ของ-competition-vqa-ตรงนี้"      # << แก้: slug ท้าย URL เช่น kaggle.com/competitions/titanic -> "titanic"  (ใช้เฉพาะวิธี A)
KAGGLE_USERNAME = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "peetwan1"  (บน Kaggle Notebook เว้นว่างได้)
KAGGLE_KEY      = ""   # << แก้ (วิธี A, เฉพาะ Colab/เครื่อง) เช่น "0a1b2c3d4e5f..." (คีย์ยาว ~32 ตัวจาก kaggle.json)
# ----- วิธี B/C: อัปโหลดเอง / ต่อ Google Drive -----
DATA_DIR = ""          # << แก้: ถ้าอัปโหลดข้อมูลเอง ใส่ path โฟลเดอร์ เช่น "/content" (วิธี B) หรือ "/content/drive/MyDrive/data" (วิธี C) ; ใช้ Kaggle (วิธี A) ปล่อยว่าง

# เช็ค GPU (ถ้าไม่มี + งานเป็น deep learning -> เปิดที่เมนู Runtime > Change runtime type > T4 GPU)
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "ไม่มี (งาน deep learning จะช้า; งานตาราง/ตัดคำ ไม่ต้องใช้ GPU ก็ได้)")
except Exception:
    pass

def get_data(comp):
    # วิธี B/C: ผู้ใช้ตั้ง DATA_DIR เอง (อัปโหลด/ต่อ Drive) -- แตก zip ในโฟลเดอร์นั้นให้ด้วยถ้ามี
    if DATA_DIR and os.path.isdir(DATA_DIR):
        for z in glob.glob(os.path.join(DATA_DIR, "*.zip")):
            subprocess.run(["unzip", "-o", "-q", z, "-d", DATA_DIR])
        print("ใช้ข้อมูลจากโฟลเดอร์ที่ตั้งเอง:", DATA_DIR); return DATA_DIR
    # บน Kaggle Notebook ข้อมูลต่อไว้แล้ว
    kpath = f"/kaggle/input/{comp}"
    if os.path.isdir(kpath):
        print("อยู่บน Kaggle ข้อมูลอยู่ที่", kpath); return kpath
    # วิธี A: ดาวน์โหลดด้วย Kaggle API
    if KAGGLE_USERNAME and KAGGLE_KEY:
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        open(os.path.expanduser("~/.kaggle/kaggle.json"), "w").write(
            '{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
        os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    os.makedirs("data", exist_ok=True)
    subprocess.run(["kaggle","competitions","download","-c",comp,"-p","data"], check=True)
    for z in glob.glob("data/*.zip"):
        subprocess.run(["unzip","-o","-q",z,"-d","data"], check=True)
    return "data"

DATA_DIR = get_data(COMP)
print("\nไฟล์ที่มี (ดูชื่อไฟล์/โฟลเดอร์ แล้วแก้ในเซลล์ถัดไปให้ตรง):")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "**"), recursive=True))[:40]:
    print("  ", p)

## ขั้นที่ 3 — path + CONFIG

In [ ]:
import os, glob, pandas as pd, torch
from PIL import Image
TEST_IMG = os.path.join(DATA_DIR,"test")            # << แก้: โฟลเดอร์รูป test เช่น os.path.join(DATA_DIR,"images","test")
SAMPLE_SUB = os.path.join(DATA_DIR,"sample_submission.csv")
test = pd.read_csv(os.path.join(DATA_DIR,"test.csv"))   # << แก้: ถ้าไฟล์ไม่ชื่อ test.csv เช่น "test_questions.csv"
sample = pd.read_csv(SAMPLE_SUB)
IMG_ID_COL = "image"     # << แก้: ชื่อคอลัมน์ไฟล์รูปใน test (ดูจาก "test คอลัมน์:") เช่น "image_id", "filename"
Q_COL      = "question"  # << แก้: ชื่อคอลัมน์คำถาม เช่น "question_th" (ถ้าเป็น OCR ล้วน ไม่มีคำถาม ตั้งเป็น None)
ANS_COL    = sample.columns[1]
print("test คอลัมน์:",list(test.columns)); display(test.head()); display(sample.head())
# ตรวจชื่อคอลัมน์ก่อน กันพังกลางทาง
assert IMG_ID_COL in test.columns, f'ไม่มีคอลัมน์ {IMG_ID_COL} -- เลือกจาก {list(test.columns)}'
assert (Q_COL is None) or (Q_COL in test.columns), f'ไม่มีคอลัมน์ {Q_COL} -- เลือกจาก {list(test.columns)}'

## 🔎 โจทย์นี้ต้องส่งอะไร + วัดด้วยอะไร (รันก่อนทำ submission)

เซลล์นี้ตอบ 3 คำถามที่มือใหม่มักไม่รู้:
- ต้องเติม "คอลัมน์อะไร" ลง submission และเป็น "ชนิดไหน" (ดูจาก sample_submission)
- โจทย์วัดด้วย "metric อะไร" (ดึงจาก Kaggle ให้อัตโนมัติ)
- ต้องส่งเป็น "ป้าย / ความน่าจะเป็น / ตัวเลข" แบบไหน

In [ ]:
# บอกว่าต้องเติมอะไรลง submission + ตัวอย่างค่าที่ sample ให้มา
print("ไฟล์ส่งต้องมีคอลัมน์:", list(sample.columns), "| จำนวนแถว:", len(sample))
for _c in list(sample.columns)[1:]:
    print(f"  - เติม '{_c}': ชนิด {sample[_c].dtype}, ตัวอย่างค่าใน sample = {list(sample[_c].dropna().unique()[:3])}")
# ดึง metric จาก Kaggle อัตโนมัติ (ถ้าต่อ API ได้)
_metric = None
try:
    from kaggle.api.kaggle_api_extended import KaggleApi
    _api = KaggleApi(); _api.authenticate()
    for _co in _api.competitions_list(search=COMP):
        if str(getattr(_co, "ref", "")).rstrip("/").split("/")[-1] == COMP:
            _metric = getattr(_co, "evaluationMetric", None) or getattr(_co, "evaluation_metric", None); break
except Exception:
    pass
def _how_to_send(m):
    m = (m or "").lower()
    if any(k in m for k in ["auc","roc","log loss","logloss","brier","probab"]): return "ความน่าจะเป็น (เลข 0-1)"
    if any(k in m for k in ["rmse","mae","mse","r2","rmsle","error","mape","smape"]): return "ตัวเลขต่อเนื่อง"
    return "ป้าย/คลาส (เช่น 0/1 หรือชื่อคลาส)"
print("\nMetric (ดึงจาก Kaggle):", _metric or "ดึงไม่ได้ -> เปิด tab Evaluation บนหน้า Kaggle อ่านเอง")
print("=> ปกติต้องส่งเป็น:", _how_to_send(_metric), "(เช็คกับ tab Evaluation อีกที)")

## วิธีที่ 1 — BLIP-VQA สำเร็จรูป (ง่ายสุด ไม่ต้องเทรน)

ตอบคำถามจากรูปแบบ zero-shot สำหรับภาษาไทยที่ซับซ้อนแนะนำเปลี่ยนเป็น Qwen2.5-VL (ดูคอมเมนต์)

In [ ]:
from transformers import pipeline
from tqdm.auto import tqdm
dev = 0 if torch.cuda.is_available() else -1
print("เตือน: BLIP ตอบเป็นภาษาอังกฤษ + อ่านตัวอักษร(OCR)ไม่เก่ง -- ถ้าโจทย์ต้องการคำตอบไทย/OCR ให้ใช้ Qwen2.5-VL เซลล์ถัดไปแทน")
vqa = pipeline("visual-question-answering", model="Salesforce/blip-vqa-base", device=dev)
ans=[]
for _,r in tqdm(test.iterrows(), total=len(test)):   # มีแถบ progress ให้เห็นว่าไม่ค้าง
    img = Image.open(os.path.join(TEST_IMG, str(r[IMG_ID_COL]))).convert("RGB")
    q = str(r[Q_COL]) if Q_COL else "What is in the image?"
    ans.append(vqa(image=img, question=q, top_k=1)[0]["answer"])
out=sample.copy(); out[ANS_COL]=ans
out.to_csv("submission.csv",index=False,encoding="utf-8-sig")
print("บันทึก submission.csv"); display(out.head())

## ทางเลือก — Qwen2.5-VL (รองรับไทย + OCR ดีกว่า, ต้อง GPU)

ปลดคอมเมนต์ถ้าต้องการคำตอบภาษาไทยหรืออ่านตัวอักษรในรูป

In [ ]:
# from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
# mdl = Qwen2_5_VLForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct",
#         torch_dtype="auto", device_map="auto")
# proc = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")
# def ask(img_path, question):
#     msgs=[{"role":"user","content":[{"type":"image","image":img_path},{"type":"text","text":question}]}]
#     text=proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
#     inputs=proc(text=[text], images=[Image.open(img_path).convert("RGB")], return_tensors="pt").to(mdl.device)
#     out=mdl.generate(**inputs, max_new_tokens=64)
#     return proc.batch_decode(out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
print("ปลดคอมเมนต์เซลล์นี้ถ้าจะใช้ Qwen2.5-VL (ภาษาไทย/OCR)")

## ขั้นสุดท้าย — ส่ง submission ขึ้น Kaggle

เช็คก่อนส่งทุกครั้ง: ไฟล์ submission ต้องมีชื่อคอลัมน์ + จำนวนแถว ตรงกับ `sample_submission.csv` เป๊ะ ๆ
- บน Kaggle Notebook: กดปุ่ม `Submit` ที่แถบขวา (ง่ายสุด) หรือใช้คำสั่งข้างล่าง
- บน Colab/เครื่อง: รันเซลล์ข้างล่าง (ต้องตั้ง token แล้ว)

In [ ]:
import pandas as pd, glob, os
FILE = "submission.csv"   # << แก้เป็นไฟล์ที่จะส่ง เช่น "submission_lgbm.csv" หรือ "submission_timm.csv"
# ตรวจรูปแบบก่อนส่งอัตโนมัติ (กันแก้ config ผิดแล้วส่งฟอร์แมตเพี้ยน -> ได้ 0 คะแนน)
_sub = pd.read_csv(FILE)
_sam = glob.glob(os.path.join(DATA_DIR, "*ample*ubmission*.csv"))
if _sam:
    _s = pd.read_csv(_sam[0])
    print("คอลัมน์ตรง sample:", list(_sub.columns)==list(_s.columns), "| จำนวนแถวตรง:", len(_sub)==len(_s))
    assert list(_sub.columns)==list(_s.columns), f"คอลัมน์ไม่ตรง sample! {list(_sub.columns)} != {list(_s.columns)} -> แก้ก่อนส่ง"
print("พร้อมส่ง:", FILE, _sub.shape)
!kaggle competitions submit -c {COMP} -f {FILE} -m "blip vqa zero-shot"
!kaggle competitions submissions -c {COMP} | head